# Supporting Data Preparation

This notebook prepares every file that `FINAL.ipynb` depends on. Run it once (top-to-bottom) before opening `FINAL.ipynb`.

**Files produced:**

| File | Produced by | Used in FINAL.ipynb |
|------|-------------|---------------------|
| `data/cache/matched_906.pkl` | Part 1 §1 | §2 cohort load |
| `data/1718/raw/synapses_matched.csv` | Part 1 §2 | §2 connectivity matrix |
| `data/G_906.pkl` / `data/G_93.pkl` | Part 1 §3 | Part 2 (3D viz + export) |
| `data/exports/G_906_nodes.csv` / `G_906_edges.csv` | Part 2 | §3 pair table |
| `data/exports/G_93_nodes.csv` / `G_93_edges.csv` | Part 2 | §9.1 cross-check |
| `outputs/functional_network/F_correlation_matrix.npy` | Part 3 | §9.1 cross-check |
| `outputs/functional_network/functional_cohort.csv` | Part 3 | §9.1 cross-check |

**Sources:** `KevinStructural.ipynb` · `visualize_structural.ipynb` · `functional_network.ipynb`

---
# Part 1 — Structural Network Construction

Builds two weighted directed graphs (906-neuron full cohort and 93-neuron L4-V1 scan-9.3 subnetwork) from the MICrONS synapse table. Exports the cohort DataFrame, synapse CSV, and both NetworkX graph pickles.

## §1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import pickle
import os

import microns_datacleaner as mic
import microns_datacleaner.filters as fl

## §2 — Load and filter neurons

Filters to `ax_clean` proofread + functionally matched neurons (906 total). Then isolates the session-9, scan-3, V1 subset (93 neurons, all L4). Saves `matched_906.pkl` so `FINAL.ipynb` can load the cohort without re-running the datacleaner API.

In [ ]:
cleaner = mic.MicronsDataCleaner(datadir="data", version=1718, download_policy='minimum')
units, _ = cleaner.process_nucleus_data(functional_data='best_only')

proofread = fl.filter_neurons(units, proofread='ax_clean')
matched   = fl.filter_neurons(proofread, tuning='matched').reset_index(drop=True)

print(f"Full working set: {len(matched)} neurons")

In [ ]:
# Session 9.3 V1 only: session=9, scan_idx=3, brain_area='V1'
sess93_v1 = matched[
    (matched['session']    == 9) &
    (matched['scan_idx']   == 3) &
    (matched['brain_area'] == 'V1')
].reset_index(drop=True)

print(f"Session 9.3 V1 neurons: {len(sess93_v1)}")
print(f"Layer breakdown:     {sess93_v1['layer'].value_counts().to_dict()}")
print(f"Cell type breakdown: {sess93_v1['cell_type'].value_counts().to_dict()}")

In [ ]:
# Save the 906-neuron cohort so FINAL.ipynb can load it from cache
import pathlib
pathlib.Path('data/cache').mkdir(parents=True, exist_ok=True)
matched.to_pickle('data/cache/matched_906.pkl')
print("Saved: data/cache/matched_906.pkl")

## §3 — Load synapses

Builds `synapses_matched.csv` and filters to the two subnetworks. Edge weight = synaptic cleft area in voxels (proxy for synaptic strength).

In [ ]:
# Build synapses_matched.csv from the raw connection tables.
cleaner.merge_synapses('synapses_matched')

synapses = pd.read_csv("data/1718/raw/synapses_matched.csv")
synapses = synapses[synapses['pre_pt_root_id'] != synapses['post_pt_root_id']].reset_index(drop=True)

# Cast 18-digit IDs to int64 to avoid float64 precision loss
synapses['pre_pt_root_id']  = synapses['pre_pt_root_id'].astype('int64')
synapses['post_pt_root_id'] = synapses['post_pt_root_id'].astype('int64')
matched['pt_root_id']       = matched['pt_root_id'].astype('int64')

print(f"Full synapse table:   {len(synapses):,} connections")
print(f"Size range:           {synapses['size'].min():.0f} – {synapses['size'].max():.0f} voxels")
print()

ids_93 = set(sess93_v1['pt_root_id'].astype('int64').tolist())
syn_93 = synapses[
    synapses['pre_pt_root_id'].isin(ids_93) &
    synapses['post_pt_root_id'].isin(ids_93)
].reset_index(drop=True)

print(f"Session 9.3 V1 table: {len(syn_93):,} connections")

## §4 — Build weighted directed graphs

`nx.DiGraph` nodes carry: `layer`, `cell_type`, `brain_area`, `pref_ori`, `gOSI`, spatial position. Edge `weight` = synapse size (voxels).

In [ ]:
def build_weighted_digraph(neurons_df, syn_df):
    G = nx.DiGraph()
    node_ids    = neurons_df['pt_root_id'].astype('int64').tolist()
    layers      = neurons_df['layer'].tolist()
    cell_types  = neurons_df['cell_type'].tolist()
    brain_areas = neurons_df['brain_area'].tolist()
    pref_oris   = neurons_df['pref_ori'].tolist()
    gOSIs       = neurons_df['gOSI'].tolist()
    xs          = neurons_df['pt_position_x'].tolist()
    ys          = neurons_df['pt_position_y'].tolist()
    zs          = neurons_df['pt_position_z'].tolist()

    for i, nid in enumerate(node_ids):
        G.add_node(nid, layer=layers[i], cell_type=cell_types[i],
                   brain_area=brain_areas[i], pref_ori=pref_oris[i],
                   gOSI=gOSIs[i], x=xs[i], y=ys[i], z=zs[i])

    node_set = set(G.nodes())
    pre_ids  = syn_df['pre_pt_root_id'].astype('int64').tolist()
    post_ids = syn_df['post_pt_root_id'].astype('int64').tolist()
    weights  = syn_df['size'].tolist()

    edges_added = 0
    for pre, post, w in zip(pre_ids, post_ids, weights):
        if pre in node_set and post in node_set:
            G.add_edge(pre, post, weight=w)
            edges_added += 1

    print(f"  Nodes: {G.number_of_nodes()}, edges added: {edges_added}")
    return G

print("Building 906-neuron network...")
G_906 = build_weighted_digraph(matched, synapses)

print("Building 93-neuron network...")
G_93  = build_weighted_digraph(sess93_v1, syn_93)

print()
print(f"Full 906-neuron network:     {G_906.number_of_nodes()} nodes, {G_906.number_of_edges():,} edges, density={nx.density(G_906):.4f}")
print(f"Session 9.3 V1 network (93): {G_93.number_of_nodes()}  nodes, {G_93.number_of_edges():,} edges,  density={nx.density(G_93):.4f}")

## §5 — Degree and strength statistics

In [ ]:
def compute_stats(G):
    out_deg = np.array([d for _, d in G.out_degree()])
    in_deg  = np.array([d for _, d in G.in_degree()])
    out_str = np.array([d for _, d in G.out_degree(weight='weight')])
    in_str  = np.array([d for _, d in G.in_degree(weight='weight')])
    return out_deg, in_deg, out_str, in_str

out_deg_906, in_deg_906, out_str_906, in_str_906 = compute_stats(G_906)
out_deg_93,  in_deg_93,  out_str_93,  in_str_93  = compute_stats(G_93)

def print_stats(label, out_deg, in_deg, out_str, in_str):
    print(f"\n{'='*56}\n {label}\n{'='*56}")
    print(f"  {'Metric':<35} {'Out':>9} {'In':>9}")
    print(f"  {'-'*53}")
    print(f"  {'Mean degree':<35} {out_deg.mean():>9.2f} {in_deg.mean():>9.2f}")
    print(f"  {'Max degree':<35} {out_deg.max():>9} {in_deg.max():>9}")
    print(f"  {'Neurons with degree 0':<35} {(out_deg==0).sum():>9} {(in_deg==0).sum():>9}")
    print(f"  {'Mean strength (voxels)':<35} {out_str.mean():>9.0f} {in_str.mean():>9.0f}")
    print(f"  {'Max strength (voxels)':<35} {out_str.max():>9.0f} {in_str.max():>9.0f}")

print_stats('Full 906-neuron Network',     out_deg_906, in_deg_906, out_str_906, in_str_906)
print_stats('Session 9.3 V1 Network (93)', out_deg_93,  in_deg_93,  out_str_93,  in_str_93)

## §6 — Strength and degree distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

total_deg_906 = out_deg_906 + in_deg_906
axes[0,0].hist(total_deg_906, bins=40, color='tomato', edgecolor='white', alpha=0.85)
axes[0,0].axvline(total_deg_906.mean(), color='darkred', lw=2, linestyle='--',
                  label=f'mean = {total_deg_906.mean():.1f}')
axes[0,0].set(xlabel='Total Degree (in + out)', ylabel='Count',
              title='906-neuron: Degree Distribution')
axes[0,0].legend()

total_str_906 = out_str_906 + in_str_906
axes[0,1].hist(total_str_906, bins=40, color='tomato', edgecolor='white', alpha=0.85)
axes[0,1].axvline(total_str_906.mean(), color='darkred', lw=2, linestyle='--',
                  label=f'mean = {total_str_906.mean():.0f}')
axes[0,1].set(xlabel='Total Strength (sum of synapse sizes)', ylabel='Count',
              title='906-neuron: Strength Distribution')
axes[0,1].legend()

m = max(out_str_906.max(), in_str_906.max())
axes[0,2].scatter(out_str_906, in_str_906, alpha=0.4, s=15, color='tomato')
axes[0,2].plot([0, m], [0, m], 'k--', alpha=0.3, label='y = x')
axes[0,2].set(xlabel='Out-strength', ylabel='In-strength',
              title='906-neuron: Out-strength vs In-strength')
axes[0,2].legend()

total_deg_93 = out_deg_93 + in_deg_93
axes[1,0].hist(total_deg_93, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[1,0].axvline(total_deg_93.mean(), color='navy', lw=2, linestyle='--',
                  label=f'mean = {total_deg_93.mean():.1f}')
axes[1,0].set(xlabel='Total Degree (in + out)', ylabel='Count',
              title='93-neuron (sess 9.3 V1): Degree Distribution')
axes[1,0].legend()

total_str_93 = out_str_93 + in_str_93
axes[1,1].hist(total_str_93, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[1,1].axvline(total_str_93.mean(), color='navy', lw=2, linestyle='--',
                  label=f'mean = {total_str_93.mean():.0f}')
axes[1,1].set(xlabel='Total Strength (sum of synapse sizes)', ylabel='Count',
              title='93-neuron (sess 9.3 V1): Strength Distribution')
axes[1,1].legend()

if out_str_93.max() > 0:
    m2 = max(out_str_93.max(), in_str_93.max())
    axes[1,2].scatter(out_str_93, in_str_93, alpha=0.6, s=40, color='steelblue')
    axes[1,2].plot([0, m2], [0, m2], 'k--', alpha=0.3, label='y = x')
    axes[1,2].legend()
axes[1,2].set(xlabel='Out-strength', ylabel='In-strength',
              title='93-neuron (sess 9.3 V1): Out-strength vs In-strength')

plt.suptitle('Structural Network — Degree and Strength Distributions', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## §7 — Synapse size distribution

Distribution of individual edge weights (cleft area, voxels). Typically right-skewed — most synapses are small with a long tail of large ones.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

weights_906 = synapses['size'].values
axes[0].hist(weights_906, bins=60, color='tomato', edgecolor='white', alpha=0.85)
axes[0].axvline(np.median(weights_906), color='darkred', lw=2, linestyle='--',
                label=f'median = {np.median(weights_906):.0f}')
axes[0].set(xlabel='Synapse Size (voxels)', ylabel='Count',
            title='906-neuron: Synapse Size Distribution')
axes[0].legend()

weights_93 = syn_93['size'].values
if len(weights_93) > 0:
    axes[1].hist(weights_93, bins=25, color='steelblue', edgecolor='white', alpha=0.85)
    axes[1].axvline(np.median(weights_93), color='navy', lw=2, linestyle='--',
                    label=f'median = {np.median(weights_93):.0f}')
    axes[1].legend()
axes[1].set(xlabel='Synapse Size (voxels)', ylabel='Count',
            title='93-neuron (sess 9.3 V1): Synapse Size Distribution')

plt.tight_layout(); plt.show()
print(f"906-network — median: {np.median(weights_906):.0f}, mean: {weights_906.mean():.0f}, skewness: {pd.Series(weights_906).skew():.2f}")
if len(weights_93) > 0:
    print(f"93-network  — median: {np.median(weights_93):.0f},  mean: {weights_93.mean():.0f},  skewness: {pd.Series(weights_93).skew():.2f}")

## §8 — 2D network visualizations

**93-neuron:** Spring layout, edge thickness ∝ synapse size (log-scaled).  
**906-neuron:** Actual x/z tissue coordinates. Node size ∝ total strength. Above-median edges only.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
np.random.seed(42)
pos_93 = nx.spring_layout(G_93, weight='weight', seed=42, k=0.5)

cell_types_93 = [G_93.nodes[n]['cell_type'] for n in G_93.nodes()]
unique_types  = sorted(set(cell_types_93))
cmap_nodes    = plt.cm.Set2(np.linspace(0, 1, max(len(unique_types), 1)))
color_map     = {ct: cmap_nodes[i] for i, ct in enumerate(unique_types)}
node_colors   = [color_map[ct] for ct in cell_types_93]

edge_w_93 = np.array([G_93[u][v]['weight'] for u, v in G_93.edges()])
if len(edge_w_93) > 0:
    log_w  = np.log1p(edge_w_93)
    norm_w = (log_w - log_w.min()) / (log_w.max() - log_w.min() + 1e-9)
    widths = 0.4 + 3.5 * norm_w; alphas = 0.2 + 0.6 * norm_w
    for (u, v), w, a in zip(G_93.edges(), widths, alphas):
        nx.draw_networkx_edges(G_93, pos_93, edgelist=[(u, v)], width=float(w),
                               alpha=float(a), edge_color='gray', arrows=True,
                               arrowsize=10, connectionstyle='arc3,rad=0.1', ax=ax)

nx.draw_networkx_nodes(G_93, pos_93, node_color=node_colors, node_size=150, ax=ax)
nx.draw_networkx_labels(G_93, pos_93,
                        labels={n: str(i) for i, n in enumerate(G_93.nodes())},
                        font_size=5, ax=ax)
for ct, c in color_map.items():
    ax.scatter([], [], c=[c], s=80, label=ct)
ax.legend(title='Cell type', loc='upper left', fontsize=8)
ax.set_title(f'Session 9.3 V1 Structural Network ({G_93.number_of_nodes()} neurons, '
             f'{G_93.number_of_edges()} synapses)\nEdge thickness ∝ synapse size | All L4 neurons')
ax.axis('off'); plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
pos_906       = {n: (G_906.nodes[n]['x'], G_906.nodes[n]['z']) for n in G_906.nodes()}
area_colors   = {'V1': 'steelblue', 'RL': 'tomato', 'AL': 'gold', 'LM': 'green'}
node_cols_906 = [area_colors.get(G_906.nodes[n]['brain_area'], 'gray') for n in G_906.nodes()]
total_str_all = np.array([G_906.degree(n, weight='weight') for n in G_906.nodes()])
node_sz       = 8 + 60 * (total_str_all / total_str_all.max())

all_w_906   = np.array([G_906[u][v]['weight'] for u, v in G_906.edges()])
heavy_edges = [(u, v) for u, v in G_906.edges() if G_906[u][v]['weight'] >= np.median(all_w_906)]
nx.draw_networkx_edges(G_906, pos_906, edgelist=heavy_edges,
                       alpha=0.06, width=0.4, edge_color='gray', arrows=False, ax=ax)
nx.draw_networkx_nodes(G_906, pos_906, node_color=node_cols_906,
                       node_size=node_sz, alpha=0.8, ax=ax)
for area, color in area_colors.items():
    ax.scatter([], [], c=color, s=60, label=area)
ax.legend(title='Brain area', loc='upper right', fontsize=9)
ax.set_title(f'Full 906-neuron Structural Network — Spatial Layout\n'
             f'Node size ∝ total strength | Above-median edges shown | {G_906.number_of_edges():,} total synapses')
ax.set_xlabel('x position (μm)'); ax.set_ylabel('z position (μm)')
plt.tight_layout(); plt.show()

## §9 — Summary table

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Nodes', 'Edges (synapses)', 'Network density',
        'Mean out-degree', 'Mean in-degree', 'Max out-degree', 'Max in-degree',
        'Mean out-strength (voxels)', 'Mean in-strength (voxels)',
        'Max out-strength (voxels)', 'Max in-strength (voxels)',
        'Median synapse size (voxels)', 'Neurons with zero output', 'Neurons with zero input',
    ],
    'Full 906-neuron': [
        G_906.number_of_nodes(), G_906.number_of_edges(), f"{nx.density(G_906):.4f}",
        f"{out_deg_906.mean():.2f}", f"{in_deg_906.mean():.2f}",
        int(out_deg_906.max()), int(in_deg_906.max()),
        f"{out_str_906.mean():.0f}", f"{in_str_906.mean():.0f}",
        f"{out_str_906.max():.0f}", f"{in_str_906.max():.0f}",
        f"{np.median(weights_906):.0f}",
        int((out_deg_906 == 0).sum()), int((in_deg_906 == 0).sum()),
    ],
    'Session 9.3 V1 (93 neurons)': [
        G_93.number_of_nodes(), G_93.number_of_edges(), f"{nx.density(G_93):.4f}",
        f"{out_deg_93.mean():.2f}", f"{in_deg_93.mean():.2f}",
        int(out_deg_93.max()), int(in_deg_93.max()),
        f"{out_str_93.mean():.0f}", f"{in_str_93.mean():.0f}",
        f"{out_str_93.max():.0f}", f"{in_str_93.max():.0f}",
        f"{np.median(weights_93):.0f}" if len(weights_93) > 0 else 'N/A',
        int((out_deg_93 == 0).sum()), int((in_deg_93 == 0).sum()),
    ]
})
print(summary.to_string(index=False))

In [ ]:
os.makedirs('data', exist_ok=True)
with open('data/G_906.pkl', 'wb') as f:
    pickle.dump(G_906, f)
with open('data/G_93.pkl', 'wb') as f:
    pickle.dump(G_93, f)
print("Saved: data/G_906.pkl  and  data/G_93.pkl")

---
# Part 2 — Structural Network 3D Visualization and CSV Export

Loads the graph pickles from Part 1, renders interactive Plotly 3D figures, and exports `G_906_nodes.csv`, `G_906_edges.csv`, `G_93_nodes.csv`, `G_93_edges.csv` to `data/exports/`. These four CSV files are the direct structural inputs consumed by `FINAL.ipynb` §3 and §9.1.

## §1 — Imports

In [ ]:
import pickle
import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import os

## §2 — Load graphs

In [ ]:
with open('data/G_906.pkl', 'rb') as f:
    G_906 = pickle.load(f)
with open('data/G_93.pkl', 'rb') as f:
    G_93 = pickle.load(f)

print(f'G_906: {G_906.number_of_nodes()} nodes, {G_906.number_of_edges():,} edges')
print(f'G_93:  {G_93.number_of_nodes()} nodes,  {G_93.number_of_edges():,} edges')

## §3 — Plot helper

`plot_3d(G, ...)` renders an interactive Plotly 3D scatter of the neurons (nodes sized by strength or degree, coloured by area / layer / orientation) with synaptic edges drawn as semi-transparent lines. Hover over any neuron to see its metadata.

In [ ]:
AREA_COL  = {'V1': '#4E79A7', 'RL': '#F28E2B', 'AL': '#59A14F',
             'LM': '#E15759', 'unknown': '#aaaaaa'}
LAYER_COL = {'L2/3': '#76B7B2', 'L4': '#EDC948', 'L5': '#B07AA1',
             'L6': '#FF9DA7', 'unknown': '#aaaaaa'}

def plot_3d(G, title, color_attr='brain_area', size_by='strength',
            edge_pct_keep=50, dark=True):
    nodes = list(G.nodes())
    xs = np.array([G.nodes[n]['x'] for n in nodes], dtype=float)
    ys = np.array([G.nodes[n]['y'] for n in nodes], dtype=float)
    zs = np.array([G.nodes[n]['z'] for n in nodes], dtype=float)

    if size_by == 'strength':
        raw = np.array([G.degree(n, weight='weight') for n in nodes], dtype=float)
    else:
        raw = np.array([G.degree(n) for n in nodes], dtype=float)
    norm_sz = (raw - raw.min()) / (raw.max() - raw.min() + 1e-9)
    sizes   = 4 + 16 * np.sqrt(norm_sz)

    raw_attr = [G.nodes[n].get(color_attr) for n in nodes]
    if color_attr == 'pref_ori':
        cvals = np.array([float(v) if v is not None else 0.0 for v in raw_attr])
        marker = dict(
            size=sizes, color=cvals, colorscale='HSV', cmin=0, cmax=180,
            colorbar=dict(title='Pref. ori. (deg)', thickness=14,
                          tickvals=[0, 45, 90, 135, 180],
                          tickfont=dict(color='white')),
            line=dict(width=0.8, color='rgba(255,255,255,0.5)'), opacity=0.92,
        )
        palette = {}
    else:
        palette = dict(AREA_COL if color_attr == 'brain_area' else LAYER_COL)
        unique_vals = sorted(set(str(v) for v in raw_attr))
        auto = px.colors.qualitative.Plotly
        for i, v in enumerate(unique_vals):
            if v not in palette:
                palette[v] = auto[i % len(auto)]
        col_list = [palette.get(str(v), '#aaaaaa') for v in raw_attr]
        marker = dict(size=sizes, color=col_list,
                      line=dict(width=0.8, color='rgba(255,255,255,0.5)'), opacity=0.92)

    hover = []
    for n in nodes:
        d = G.nodes[n]
        st = G.degree(n, weight='weight'); dg = G.degree(n)
        ori = d.get('pref_ori')
        ori_s = f'{float(ori):.1f}' if ori is not None else 'N/A'
        hover.append(f'<b>Neuron {n}</b><br>'
                     f'Area: {d.get("brain_area","?")}  Layer: {d.get("layer","?")}'
                     f'  Type: {d.get("cell_type","?")}<br>'
                     f'Pref ori: {ori_s} deg  gOSI: {float(d.get("gOSI",0)):.3f}<br>'
                     f'<b>Degree: {dg}   Strength: {st:,}</b>')

    node_trace = go.Scatter3d(x=xs, y=ys, z=zs, mode='markers',
                              marker=marker, text=hover, hoverinfo='text',
                              name='Neurons', showlegend=False)

    legend_traces = []
    if palette:
        for val in sorted(palette):
            legend_traces.append(go.Scatter3d(
                x=[None], y=[None], z=[None], mode='markers',
                marker=dict(size=10, color=palette[val]),
                name=str(val), showlegend=True))

    all_edges = list(G.edges(data=True))
    edge_traces = []
    if all_edges:
        ew = np.array([d['weight'] for _, _, d in all_edges], dtype=float)
        threshold = np.percentile(ew, 100 - edge_pct_keep)
        sel = [(u, v, d) for u, v, d in all_edges if d['weight'] >= threshold]
        ew_sel  = np.array([d['weight'] for _, _, d in sel], dtype=float)
        ew_norm = (ew_sel - ew_sel.min()) / (ew_sel.max() - ew_sel.min() + 1e-9)
        N = 5
        buckets = [[] for _ in range(N)]
        for idx, (u, v, _) in enumerate(sel):
            b = min(int(ew_norm[idx] * N), N - 1)
            buckets[b].append((u, v))
        for b, elist in enumerate(buckets):
            if not elist:
                continue
            alpha = 0.05 + 0.25 * (b / (N - 1))
            ex, ey, ez = [], [], []
            for u, v in elist:
                ex += [G.nodes[u]['x'], G.nodes[v]['x'], None]
                ey += [G.nodes[u]['y'], G.nodes[v]['y'], None]
                ez += [G.nodes[u]['z'], G.nodes[v]['z'], None]
            edge_traces.append(go.Scatter3d(
                x=ex, y=ey, z=ez, mode='lines',
                line=dict(color=f'rgba(180,180,220,{alpha:.2f})', width=1),
                hoverinfo='none', showlegend=(b == N - 1),
                name=f'Synapses ({len(sel):,} shown)'))

    bg = '#0e0e12' if dark else 'white'; axcol = '#888888' if dark else '#333'; tcol = 'white' if dark else 'black'
    fig = go.Figure(data=edge_traces + legend_traces + [node_trace])
    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=19, color=tcol, family='Arial')),
        paper_bgcolor=bg,
        scene=dict(bgcolor=bg,
                   xaxis=dict(title='x (nm)', showgrid=False, zeroline=False, color=axcol, showbackground=False),
                   yaxis=dict(title='y (nm)', showgrid=False, zeroline=False, color=axcol, showbackground=False),
                   zaxis=dict(title='z (nm)', showgrid=False, zeroline=False, color=axcol, showbackground=False),
                   camera=dict(eye=dict(x=1.4, y=1.4, z=0.8))),
        legend=dict(font=dict(color=tcol, size=12), bgcolor='rgba(0,0,0,0.35)',
                    bordercolor='#444', borderwidth=1, itemsizing='constant'),
        margin=dict(l=0, r=0, b=0, t=60), height=720)
    return fig

## §4 — Interactive 3D plots

### Full 906-neuron network — coloured by brain area
Node size ∝ total synaptic strength. Top 50 % of edges by weight drawn.

In [ ]:
fig_906 = plot_3d(G_906,
    title='Full 906-Neuron Structural Network  |  Colour: Brain Area',
    color_attr='brain_area', size_by='strength', edge_pct_keep=50)
fig_906.show()

### Full 906-neuron network — coloured by cortical layer

In [ ]:
fig_906_layer = plot_3d(G_906,
    title='Full 906-Neuron Network  |  Colour: Cortical Layer',
    color_attr='layer', size_by='degree', edge_pct_keep=30)
fig_906_layer.show()

### Session 9.3 V1 subnetwork (93 neurons)

All neurons are L4 in V1. Colour = preferred orientation (cyclic HSV, 0°–180°). All 229 synapses shown.

In [ ]:
fig_93 = plot_3d(G_93,
    title='Session 9.3 V1 Network (93 neurons)  |  Colour: Preferred Orientation',
    color_attr='pref_ori', size_by='degree', edge_pct_keep=100)
fig_93.show()

In [ ]:
fig_93b = plot_3d(G_93,
    title='Session 9.3 V1 Network  |  Colour: Brain Area  |  Size: Strength',
    color_attr='brain_area', size_by='strength', edge_pct_keep=100)
fig_93b.show()

## §5 — Export node and edge tables as CSV

**This is the critical output step.** The four CSV files saved here are loaded directly by `FINAL.ipynb` §3 (pair table) and §9.1 (93-neuron cross-check).

In [ ]:
os.makedirs('data/exports', exist_ok=True)

def export_graph(G, prefix):
    rows = []
    for n in G.nodes():
        d = G.nodes[n]
        rows.append({
            'neuron_id':    n,
            'brain_area':   d.get('brain_area'),
            'layer':        d.get('layer'),
            'cell_type':    d.get('cell_type'),
            'pref_ori':     d.get('pref_ori'),
            'gOSI':         d.get('gOSI'),
            'x':            d.get('x'),
            'y':            d.get('y'),
            'z':            d.get('z'),
            'out_degree':   G.out_degree(n),
            'in_degree':    G.in_degree(n),
            'out_strength': G.out_degree(n, weight='weight'),
            'in_strength':  G.in_degree(n, weight='weight'),
        })
    nodes_df = pd.DataFrame(rows)
    nodes_df.to_csv(f'data/exports/{prefix}_nodes.csv', index=False)
    print(f'Saved data/exports/{prefix}_nodes.csv  ({len(nodes_df)} rows)')

    edges_df = pd.DataFrame(
        [(u, v, G[u][v]['weight']) for u, v in G.edges()],
        columns=['pre_neuron_id', 'post_neuron_id', 'synapse_size'])
    edges_df.to_csv(f'data/exports/{prefix}_edges.csv', index=False)
    print(f'Saved data/exports/{prefix}_edges.csv  ({len(edges_df)} rows)')

export_graph(G_906, 'G_906')
export_graph(G_93,  'G_93')

## §6 — Quick stats recap

In [ ]:
for label, G in [('906-neuron', G_906), ('93-neuron (sess 9.3 V1)', G_93)]:
    od  = np.array([d for _, d in G.out_degree()])
    id_ = np.array([d for _, d in G.in_degree()])
    os_ = np.array([d for _, d in G.out_degree(weight='weight')])
    is_ = np.array([d for _, d in G.in_degree(weight='weight')])
    print(f"{'='*52}\n {label}\n{'='*52}")
    print(f'  Nodes:          {G.number_of_nodes()}')
    print(f'  Edges:          {G.number_of_edges():,}')
    print(f'  Density:        {nx.density(G):.4f}')
    print(f'  Mean out-deg:   {od.mean():.2f}   Max: {od.max()}')
    print(f'  Mean in-deg:    {id_.mean():.2f}   Max: {id_.max()}')
    print(f'  Mean out-str:   {os_.mean():,.0f}   Max: {os_.max():,.0f}')
    print(f'  Mean in-str:    {is_.mean():,.0f}   Max: {is_.max():,.0f}')
    print()

---
# Part 3 — Functional Network Construction

Reads the MICrONS functional HDF5 for scan 9_3, extracts deconvolved calcium traces for the 93-neuron V1 cohort, computes the pairwise Pearson **trace-correlation matrix**, and saves it alongside the cohort metadata. These two files are loaded by `FINAL.ipynb` §9.1 (same-scan cross-check).

## §1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import microns_datacleaner as mic
import microns_datacleaner.filters as fl

## §2 — Load cohort and filter to scan 9.3 V1

In [ ]:
cleaner = mic.MicronsDataCleaner(datadir="data/", version=1718, download_policy='minimum')
units, segments = cleaner.process_nucleus_data(functional_data='best_only')

matched = units[units['tuning_type'] == 'matched']
proofread_matched = matched[matched['strategy_axon'] != 'none']

print(f"Total cells: {len(units)}")
print(f"Functionally matched: {len(matched)}")
print(f"Matched + proofread: {len(proofread_matched)}")

In [ ]:
SESSION = 9
SCAN = 3

cohort = proofread_matched[
    (proofread_matched['session'] == SESSION) &
    (proofread_matched['scan_idx'] == SCAN)
].reset_index(drop=True)

cohort['matrix_index'] = np.arange(len(cohort))
unit_ids = cohort['unit_id'].astype(int).values

print(f"Cohort: {len(cohort)} neurons in session {SESSION}, scan {SCAN}")
print(f"unit_ids range: {unit_ids.min()} to {unit_ids.max()}")

## §3 — Read functional HDF5

Loads the `microns_functional.h5` file and extracts the V1 unit-ID → row mapping for this scan session.

In [ ]:
import h5py

sesh_scan = f"{SESSION}_{SCAN}"

with h5py.File("data/functional/microns_functional.h5", "r") as f:
    all_unit_ids = f[f"sessions/{sesh_scan}/meta/unit_ids"][:]
    v1_indices   = f[f"sessions/{sesh_scan}/meta/area_indices/V1"][:]

v1_unit_ids  = all_unit_ids[v1_indices]
unit_id_to_row = {int(uid): i for i, uid in enumerate(v1_unit_ids)}
print(f"V1 unit_ids in scan {sesh_scan}: {len(v1_unit_ids)} neurons")

## §4 — Extract cohort traces and compute trial responses

In [ ]:
funcreader = mic.MicronsFunctionalReader(path="data/functional/microns_functional.h5")
print("Connected to functional data")

In [ ]:
trial_responses = []
trial = 0

while True:
    try:
        r = funcreader.get_trial(sesh_scan, trial, "V1")
        trial_responses.append(r['responses'])
        trial += 1
    except Exception as e:
        print(f"Stopped at trial {trial}: {type(e).__name__}")
        break

print(f"Found {trial} trials in scan {sesh_scan}")
if trial_responses:
    print(f"First trial shape (all V1 neurons × frames): {trial_responses[0].shape}")

In [ ]:
cohort = cohort[cohort['unit_id'].isin(unit_id_to_row)].reset_index(drop=True)
cohort['matrix_index'] = np.arange(len(cohort))
cohort_rows = np.array([unit_id_to_row[int(uid)] for uid in cohort['unit_id']])
print(f"Final cohort: {len(cohort)} neurons")

In [ ]:
my_traces = [resp[cohort_rows, :] for resp in trial_responses]
traces = np.concatenate(my_traces, axis=1)
print(f"Traces shape: {traces.shape}  ({traces.shape[0]} neurons × {traces.shape[1]} time frames)")

trace_stds = traces.std(axis=1)
n_dead = (trace_stds < 1e-6).sum()
print(f"Neurons with near-zero variance: {n_dead}")

## §5 — Example trace plots

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 8), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(traces[i, :3000], linewidth=0.7)
    ax.set_ylabel(f'Neuron {i}', rotation=0, ha='right', va='center')
axes[-1].set_xlabel('Time frame')
plt.suptitle(f'First 5 neurons, first 3000 frames (scan {sesh_scan})')
plt.tight_layout(); plt.show()

## §6 — Compute trace-correlation matrix

`F = np.corrcoef(traces)` — Pearson correlation across all concatenated trial frames. This is the **trace-correlation matrix** (signal + noise mixed). Shape: 93 × 93.

In [ ]:
F = np.corrcoef(traces)
print(f"Functional matrix shape: {F.shape}")
print(f"Diagonal (should all be 1.0): {F.diagonal()[:5]}")
print(f"Symmetric? F[5,10]={F[5,10]:.4f}, F[10,5]={F[10,5]:.4f}")
print(f"Any NaNs? {np.isnan(F).any()}")

In [ ]:
upper = F[np.triu_indices_from(F, k=1)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(upper, bins=100, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', alpha=0.5, label='Zero')
ax.axvline(upper.mean(), color='orange', linestyle='-', alpha=0.7, label=f'Mean={upper.mean():.3f}')
ax.set(xlabel='Pearson correlation', ylabel='Number of pairs',
       title=f'Pairwise trace correlations: {len(cohort)} neurons, {len(upper)} pairs')
ax.legend(); plt.tight_layout(); plt.show()

print(f"Mean: {upper.mean():.4f}  Median: {np.median(upper):.4f}  Std: {upper.std():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(F, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
plt.colorbar(im, ax=ax, label='Pearson trace correlation')
ax.set(title=f'Functional trace-correlation matrix ({len(cohort)} neurons, scan {sesh_scan})',
       xlabel='Neuron index', ylabel='Neuron index')
plt.tight_layout(); plt.show()

## §7 — Save outputs

Saves `F_correlation_matrix.npy` and `functional_cohort.csv` to `outputs/functional_network/`. These are the two files loaded by `FINAL.ipynb` §9.1.

In [ ]:
import os, json

out_dir = 'outputs/functional_network'
os.makedirs(out_dir, exist_ok=True)

np.save(f'{out_dir}/F_correlation_matrix.npy', F)

cohort_to_save = cohort[[
    'matrix_index', 'nucleus_id', 'pt_root_id', 'classification_system',
    'cell_type', 'layer', 'brain_area', 'strategy_axon', 'strategy_dendrite',
    'pt_position_x', 'pt_position_y', 'pt_position_z',
    'session', 'scan_idx', 'unit_id', 'cc_abs'
]].copy()
cohort_to_save.to_csv(f'{out_dir}/functional_cohort.csv', index=False)

metadata = {
    'session': int(SESSION), 'scan': int(SCAN),
    'n_neurons': int(len(cohort)), 'n_trials': int(trial),
    'n_time_frames': int(traces.shape[1]),
    'correlation_method': 'pearson (trace correlation, signal+noise mixed)',
    'mean_correlation': float(upper.mean()), 'median_correlation': float(np.median(upper)),
}
with open(f'{out_dir}/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved to {out_dir}/")
print(f"  F_correlation_matrix.npy : {F.shape}")
print(f"  functional_cohort.csv    : {len(cohort)} neurons")
print(f"  metadata.json")